# FabricMem 模式

本节将介绍 HIXL 的第 3 种传输模式：FabricMem 模式，帮助你了解该传输模式的产生背景、核心原理、配置方法以及适用场景。

本节学习大纲如下：

- FabricMem 模式的含义
- FabricMem 模式的配置方法
- FabricMem 模式的优势和限制
- FabricMem 模式的性能说明


## 1. FabricMem 模式的含义

FabricMem 模式通过 VMM（Virtual Memory Manager）统一编址，将超节点内所有 DRAM 纳入统一地址空间，使 NPU 能够通过 HCCS 链路直接访问远程内存。

### 1.1 产生背景

大模型推理场景下，KV Cache 池化是一项重要的性能优化手段，通过将 Prefill 阶段产生的 KV Cache 存储在 Host 的分布式共享 DRAM 内存中，当任意节点需要时再从 DRAM 中拉取相关的 KV Cache，从而避免重复的计算，极大地降低了推理时的首 token 时延。

KV Cache 池化技术对 D2rH 的数据传输性能提出更高的性能要求，前两章介绍的直传模式和 中转 Buffer 模式，存在传输链路功能或性能上的限制，均不能满足上述要求。因此，FabricMem 模式应运而生，其通过内存统一编址实现了 A3 场景下 D2rH 的 HCCS 直传能力，极大地提升了 KV Cache 池化场景下的数据传输性能。

### 1.2 VMM 统一编址

**FabricMem 核心机制**：

1. **物理内存虚拟化**：将各节点的物理 DRAM 映射到统一的虚拟地址空间
2. **ShareHandle 交换**：通过共享句柄让不同进程"看见"同一块内存
3. **SDMA 直通**：NPU 通过 HCCS 链路直接读写远程内存，无需 CPU 搬运

<img src="./images/fabricmem_memory_mapping.png" alt="中转Buffer开销示意" width="40%">

从本地 NPU 的片上内存直接往远程的 HOST 内存写数据的数据流向：

<img src="./images/fabricmem_d2rh_flow.png" alt="中转Buffer开销示意" width="80%">


## 2. FabricMem 模式的配置方法

### 2.1 前置条件

| 依赖项 | 版本要求 |
| --- | --- |
| HDK | 26.0 以上 |
| LingQu Computing Network（灵衢计算网络） | 1.5.0 以上 |
| CANN | 9.0 以上 |
| 硬件 | Atlas A3 训练系列产品 / Atlas A3 推理系列产品 |

### 2.2 启用方式

初始化引擎时在 `options` 中配置 `OPTION_ENABLE_USE_FABRIC_MEM`，取值为 `"1"` 表示开启（取值为 `"0"` 表示关闭）。

通过`OPTION_GLOBAL_RESOURCE_CONFIG`可配置 Fabric 虚拟内存池参数：

| 参数 | 取值范围 | 默认值 | 说明 |
| --- | --- | --- | --- |
| `fabric_memory.max_capacity` | (0, 1024] | 64 | 虚拟内存池总容量，单位 TB |
| `fabric_memory.start_address` | [40, 220] | 40 | 虚拟内存池起始地址，单位 TB |
| `fabric_memory.task_stream_num` | [1, 8] | 4 | 单个传输任务使用的并发流数量 |

参考示例如下：

```c++
std::map<AscendString, AscendString> options;
// 启用FabricMem模式（基础配置）
options[OPTION_ENABLE_USE_FABRIC_MEM] = "1";

// 启用FabricMem模式（完整配置）
options[OPTION_ENABLE_USE_FABRIC_MEM] = "1";
options[OPTION_GLOBAL_RESOURCE_CONFIG] =
R"({
"fabric_memory.max_capacity": "128",
"fabric_memory.start_address": "40",
"fabric_memory.task_stream_num": "4"
})";
```


### 2.3 配置约束

1. **配置冲突**：`OPTION_BUFFER_POOL`不能与`OPTION_ENABLE_USE_FABRIC_MEM`同时配置
2. **集群一致性**：所有节点必须配置相同的`OPTION_ENABLE_USE_FABRIC_MEM`值


## 3. FabricMem 模式的优势和限制

### 3.1 技术优势

**1. 超高传输带宽**

FabricMem 利用 HCCS 链路的高带宽特性，实现远超传统 RDMA 方案的传输性能：

| 传输方向 | 带宽(GB/s) | 对比 RoCE 性能提升 |
| --- | --- | --- |
| RH2D | ~103 | 约 5 倍 |
| D2RH | ~64 | 约 3 倍 |
| RD2D | ~155 | 约 7 倍 |
| D2RD | ~128 | 约 6 倍 |

**2. 零拷贝传输**

NPU 可直接访问远程 DRAM，避免了传统方案中 CPU 参与数据搬运的开销。

**3. 超节点内存池化**

FabricMem 将超节点内所有 DRAM 统一编址，形成分布式内存池：

- **容量扩展**：突破单节点内存容量限制，可池化多节点 DRAM
- **透明访问**：NPU 通过统一地址空间访问任意节点内存，无需感知物理位置
- **负载均衡**：KV Cache 可均匀分布在不同节点 DRAM 中，避免热点

### 3.2 适用场景

**场景一：PD 分离架构中的 KV Cache 传输**

在大模型推理的 PD 分离架构中，Prefill 阶段产生的大量 KV Cache 需要存储到 DRAM 池中，Decode 阶段需要快速加载。

**业务价值**：

- Prefill 节点的 KV Cache 可卸载到分布式 DRAM 池，释放 HBM 供计算使用
- Decode 节点按需加载 KV Cache，实现无限扩展的 KV Cache 存储
- 整体推理吞吐提升 3-5 倍，同时降低 HBM 成本

**场景二：Mooncake 集成**

Mooncake 是业界主流的 KV Cache 传输框架，FabricMem 可无缝对接：

- **零代码改造**：Mooncake 底层使用 HIXL 传输引擎，启用 FabricMem 仅需配置开关
- **带宽瓶颈突破**：传统方案受限于 RoCE 带宽（20GB/s），FabricMem 将瓶颈转移至 HCCS（100GB/s），性能提升 5 倍。

**场景三：替代中转模式**

在 A3 平台上，传统中转模式会与模型推理抢占 HBM 带宽。FabricMem 直接使用 HCCS 链路，避免带宽竞争，实现计算与传输解耦。

### 3.3 使用限制

| 限制项 | 说明 |
| --- | --- |
| 设备支持 | 仅支持 Atlas A3 训练系列产品和 Atlas A3 推理系列产品 |
| 传输范围 | 仅支持超节点内部传输，不支持跨超节点 |


## 4. FabricMem 模式的性能说明

### 4.1 基准测试结果

**测试环境**：单机 16 卡 Atlas A3 服务器

**测试数据**：DeepSeek-R1 KV 形状，单块大小约 8.8MB

**大块数据带宽**（1GB 或 2GB 单次传输）：

| 方向 | 数据大小(GB) | 延迟(μs) | 带宽(GB/s) |
| --- | --- | --- | --- |
| rH2D | 1 | 9,723 | 103 |
| rH2D | 2 | 19,388 | 103 |
| D2rH | 1 | 15,650 | 64 |
| D2rH | 2 | 31,250 | 64 |
| rD2D | 1 | 6,500 | 155 |
| rD2D | 2 | 12,929 | 155 |
| D2rD | 1 | 7,832 | 128 |
| D2rD | 2 | 15,643 | 128 |

### 4.2 性能优化建议

| 配置项 | 建议值 | 适用场景 |
| --- | --- | --- |
| `task_stream_num` | 1-2 | 小数据高频传输 |
| `task_stream_num` | 4-8 | 大数据块传输 |
| `max_capacity` | 根据实际使用量配置 | 预留足够余量 |
| `start_address` | 不同应用使用不同值 | 避免地址空间冲突 |

**总结**：FabricMem 模式是 A3 超节点内高带宽传输的首选方案，特别适合 PD 分离架构中的 KV Cache 传输场景。生产环境应优先使用 FabricMem 替代中转 Buffer 模式。


## 课后练习

本节介绍了 FabricMem 模式的核心原理、配置方法和适用场景，请根据学习内容完成以下题目进行自测。

1. （判断题）FabricMem 模式通过 VMM 统一编址，将超节点内所有 DRAM 纳入统一地址空间，使 NPU 能够直接访问远程内存。

2. （判断题）FabricMem 模式支持跨超节点的数据传输。

3. （单选题）FabricMem 模式需要哪个版本的 HDK 才能支持？
    A. 24.0 以上
    B. 25.0 以上
    C. 26.0 以上
    D. 27.0 以上

4. （单选题）关于 FabricMem 模式与 Buffer Pool 模式的关系，以下说法正确的是？
    A. 两者可以同时配置，互不影响
    B. FabricMem 模式优先级更高，会覆盖 Buffer Pool 配置
    C. 两者不能同时配置，`OPTION_BUFFER_POOL`不能与`OPTION_ENABLE_USE_FABRIC_MEM`同时使用
    D. Buffer Pool 模式是 FabricMem 的底层实现

5. （多选题）FabricMem 模式有哪些技术优势？
    A. 超高传输带宽，RH2D 可达约 103GB/s
    B. 零拷贝传输，NPU 直接访问远程 DRAM 无需 CPU 参与
    C. 超节点内存池化，突破单节点内存容量限制
    D. 支持跨超节点传输

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/03.04_answer.txt